In [1]:
import sqlite3
import pandas as pd
import joblib

conn = sqlite3.connect(r"E:\RetailIQ\retailiq.db")
df = pd.read_sql("SELECT * FROM customer_features", conn)
conn.close()

feature_cols = [
    "frequency", "repeat_order_count", "avg_days_between_orders",
    "avg_review_score", "review_count", "avg_delivery_days", "avg_delay_days",
    "treatment_voucher"
]

clf = joblib.load(r"E:\RetailIQ\models_churn.pkl")
df["predicted_churn_prob"] = clf.predict_proba(df[feature_cols].fillna(0))[:, 1]

# High-risk = top 20% predicted churn probability, AND not already a voucher recipient
threshold = df["predicted_churn_prob"].quantile(0.80)
high_risk = df[(df["predicted_churn_prob"] >= threshold) & (df["treatment_voucher"] == 0)]

n_high_risk = len(high_risk)
avg_clv = high_risk["monetary"].mean()  # adjust column name if different
avg_churn_prob = high_risk["predicted_churn_prob"].mean()

print(f"High-risk, untreated customers: {n_high_risk}")
print(f"Avg CLV in this group: {avg_clv:.2f}")
print(f"Avg predicted churn probability: {avg_churn_prob:.4f}")

High-risk, untreated customers: 18021
Avg CLV in this group: 183.75
Avg predicted churn probability: 0.7984
